# Datamine ESTIMA Process Example

This notebook demonstrates univariate grade estimation (Nearest Neighbour, Inverse Distance, Ordinary Kriging) using the **`estima`** process wrapper in `dmstudio`.

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
agent.initialize_sandbox()

## Step 2: Prepare Inputs
We copy the drillhole data and tutorial parameter files locally using the temporary `t_` prefix, and run `PROTOM` to define the prototype coordinates overlapping our samples.

In [ ]:
# Copy sample drillholes and parameter files locally in sandbox
project_folder = oScript.ActiveProject.Folder

shutil.copy(os.path.join(project_folder, '..', 'case_studies', 'grade_estimation', 'comp_holes.dmx'), 't_comp_holes.dmx')

# Copy parameter files using copy_database_files
agent.copy_database_files({
    '_vb_spar.dmx': 't_search_params.dmx',
    '_vb_epar.dmx': 't_estimation_params.dmx',
    '_vb_vpar.dmx': 't_kriging_vmodel.dmx'
})

# Generate model prototype in active folder (sandbox)
print("Generating block model prototype...")
dm_fil.protom(
    out_o='t_model_proto',
    rotmod_p=0,
    arguments=" 'N' 'Y' '5900' '4800' '-100' '10' '10' '10' '25' '45' '32'"
)
print("Setup complete.")

## Step 3: Run ESTIMA Grade Estimation
We run the `estima` process to interpolate the AU, CU, and DENSITY grades into our block model cells using Ordinary Kriging.

In [ ]:
# Run ESTIMA process in sandbox
dm_cmd.estima(
    proto_i='t_model_proto',
    in_i='t_comp_holes',
    srcparm_i='t_search_params',
    estparm_i='t_estimation_params',
    vmodparm_i='t_kriging_vmodel',
    model_o='t_grade_model_estima_result',
    val_f=['AU'],
    est_f=['AU_EST'],
    numobs_f=['AU_NS'],
    svol_f=['AU_SVOL'],
    var_f=['AU_VAR'],
    arguments="'OK'"
)

## Step 4: Verify and Load Results in Pandas
Read the estimated model cells to print descriptive grade statistics.

In [ ]:
# Read outputs using read_datamine
result_file = 't_grade_model_estima_result.dmx'
shutil.copy(result_file, 't_grade_model_estima_temp.dmx')
df_estima = agent.read_datamine('t_grade_model_estima_temp.dmx')
os.remove('t_grade_model_estima_temp.dmx')

print(f"ESTIMA block model cells: {len(df_estima)}")
print("\nESTIMA Gold (AU) Grade Summary:")
print(df_estima[df_estima['AU'] > -99]['AU'].describe())

## Step 5: Clean up Project Folder
Run this cell to clean up all copied and generated files, leaving only this notebook.

In [ ]:
# Cleanup local files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    os.remove(f)
    print(f"Removed {os.path.basename(f)}")